# Receipt OCR → Supabase Pipeline — End-to-End Test

Tests every stage of the pipeline added in the `receipt-ocr` feature branch:

| Stage | What it tests |
|---|---|
| 1 | easyOCR → flat token list from sample images |
| 2 | `receipt_parser.parse_receipt()` on those tokens |
| 3 | `POST /receipt/parse` on the Python FastAPI service |
| 4 | `POST /api/receipt/process` on the Next.js app (writes to Supabase) |

**Prerequisites**
- A `samples/` subdirectory next to this notebook containing receipt images (`.jpg`, `.jpeg`, `.png`, `.webp`)
- Python packages: `easyocr`, `requests`, `Pillow`
- Python API running locally (`uvicorn main:app` inside `python-api/`)
- Next.js dev server running locally (`npm run dev`)
- Set `CLERK_SESSION_TOKEN` below (copy from browser DevTools → Application → Cookies → `__session`)

In [5]:
import sys
#!{sys.executable} -m pip install requests Pillow

In [6]:
# ── Config ────────────────────────────────────────────────────────────────────
SAMPLES_DIR      = "samples"          # relative to this notebook
PYTHON_API_URL   = "http://localhost:8000"   # FastAPI
NEXTJS_API_URL   = "http://localhost:3000"   # Next.js dev server

# Clerk __session cookie value — needed to authenticate /api/receipt/process.
# Get it from browser DevTools → Application → Cookies → __session
CLERK_SESSION_TOKEN = ""  # <-- paste here

# easyOCR language list (add more as needed)
OCR_LANGUAGES = ["en"]

## Stage 0 — Setup

In [7]:
import sys, importlib, json, pathlib, pprint
from pathlib import Path

import requests
from PIL import Image

# ── Locate receipt_parser ─────────────────────────────────────────────────────
# Works whether running from lib/receipt-ocr/test/ or the repo root.
_NB_DIR = Path().resolve()  # directory of this notebook
_PARSER_CANDIDATES = [
    _NB_DIR.parent / "receipt_parser.py",                              # lib/receipt-ocr/
    _NB_DIR.parent.parent.parent / "lib" / "receipt-ocr" / "receipt_parser.py",  # repo root
]

import importlib.util as _ilu
_receipt_parser = None
for _p in _PARSER_CANDIDATES:
    if _p.exists():
        _spec = _ilu.spec_from_file_location("receipt_parser", _p)
        _receipt_parser = _ilu.module_from_spec(_spec)
        _spec.loader.exec_module(_receipt_parser)
        print(f"Loaded receipt_parser from: {_p}")
        break

if _receipt_parser is None:
    raise ImportError("Could not find receipt_parser.py — check PARSER_CANDIDATES paths above")

parse_receipt = _receipt_parser.parse_receipt

# ── Sample images ─────────────────────────────────────────────────────────────
SAMPLES_PATH = (_NB_DIR / SAMPLES_DIR).resolve()
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
sample_images = sorted([f for f in SAMPLES_PATH.iterdir() if f.suffix.lower() in IMAGE_EXTS])

if not sample_images:
    raise FileNotFoundError(f"No images found in {SAMPLES_PATH}. Add receipt images there and re-run.")

print(f"\nFound {len(sample_images)} sample image(s):")
for img in sample_images:
    print(f"  {img.name}")

Loaded receipt_parser from: /Users/yoonseongroh/PycharmProjects/SecretSauce/lib/receipt-ocr/receipt_parser.py

Found 20 sample image(s):
  0.jpg
  1.jpg
  10.jpg
  11.jpg
  12.jpg
  13.jpg
  14.jpg
  15.jpg
  16.jpg
  17.jpg
  18.jpg
  19.jpg
  2.jpg
  3.jpg
  4.jpg
  5.jpg
  6.JPG
  7.jpg
  8.jpg
  9.jpg


## Stage 1 — easyOCR → tokens

In [ ]:
import easyocr
import tempfile
import cv2
import numpy as np

print("Loading easyOCR model (first run downloads weights — may take a moment)...")
reader = easyocr.Reader(OCR_LANGUAGES)
print("Model ready.\n")


def _preprocess_receipt(img_path: Path) -> str:
    """Preprocess a receipt image and return the path to the processed file.

    Steps applied:
    1. Convert to greyscale.
    2. Upscale to a minimum height of 1500 px (EasyOCR accuracy drops sharply
       when font height falls below ~30 px).
    3. Apply adaptive Gaussian threshold to neutralise uneven illumination and
       the thermal-paper background gradient.

    Returns the path to a temporary PNG file containing the processed image.
    If preprocessing fails for any reason the original path is returned as-is.
    """
    try:
        img = cv2.imread(str(img_path))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Upscale if too small
        min_height = 1500
        if gray.shape[0] < min_height:
            scale = min_height / gray.shape[0]
            gray = cv2.resize(gray, None, fx=scale, fy=scale,
                              interpolation=cv2.INTER_CUBIC)

        # Adaptive threshold — handles shadows, folds, and low-contrast paper
        gray = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            blockSize=31,
            C=10,
        )

        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        cv2.imwrite(tmp.name, gray)
        return tmp.name
    except Exception:
        return str(img_path)  # graceful fallback to original


def _safe_readtext(reader, img_path: Path) -> list[str]:
    """Read text from a receipt image with preprocessing and edge-crop recovery.

    Strategy:
    1. Preprocess (greyscale + upscale + adaptive threshold) for better accuracy.
    2. Run EasyOCR on the processed image.
    3. If an OpenCV edge-crop assertion fires (EasyOCR bug: zero-size crop from a
       bounding box at the image boundary), add a 4 px white border and retry.
    4. If the retry also fails, return [] and log a warning so the pipeline
       continues rather than crashing.
    """
    processed_path = _preprocess_receipt(img_path)

    def _run(path: str) -> list[str]:
        return reader.readtext(path, detail=0)

    try:
        return _run(processed_path)
    except Exception:
        # Retry with a border to push all content away from the edge
        try:
            img = Image.open(processed_path).convert("RGB")
            padded = Image.new("RGB", (img.width + 4, img.height + 4), (255, 255, 255))
            padded.paste(img, (2, 2))
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                padded.save(tmp.name)
                return _run(tmp.name)
        except Exception as err:
            print(f"  ⚠  Could not OCR {img_path.name}: {err}")
            return []


# Map: image path → token list
ocr_results: dict[str, list[str]] = {}

for img_path in sample_images:
    tokens = _safe_readtext(reader, img_path)
    ocr_results[img_path.name] = tokens
    status = f"{len(tokens)} tokens" if tokens else "SKIPPED (OCR failed)"
    print(f"{'─'*60}")
    print(f"[{img_path.name}]  {status}")
    if tokens:
        print(tokens[:20], "..." if len(tokens) > 20 else "")


## Stage 2 — receipt_parser (local, no HTTP)

In [9]:
parsed_results: dict[str, dict] = {}

print(f"{'Image':<30} {'Store':<22} {'Total':>8}  {'Date':<12}  Items")
print("─" * 80)

for name, tokens in ocr_results.items():
    result = parse_receipt(tokens)
    parsed_results[name] = result
    total_str = f"{result['total']:.2f}" if result["total"] is not None else "None"
    date_str  = result["date"] or "None"
    print(f"{name:<30} {result['store']:<22} {total_str:>8}  {date_str:<12}  {len(result['items'])}")

print()
print("Detailed item breakdown:")
for name, result in parsed_results.items():
    print(f"\n  [{name}]  store={result['store']}")
    for item in result["items"]:
        print(f"    qty={item['quantity']}  price={item['price']:<7}  {item['name']}")
    if not result["items"]:
        print("    (no items extracted)")

Image                          Store                     Total  Date          Items
────────────────────────────────────────────────────────────────────────────────
0.jpg                          Unknown                    None  None          3
1.jpg                          Trader Joe's             840.00  2014-06-28    4
10.jpg                         SPAR                     338.16  2021-02-23    11
11.jpg                         Whole Foods               45.44  None          9
12.jpg                         Walmart                    None  None          1
13.jpg                         Walmart                   50.60  None          1
14.jpg                         Walmart                   76.60  2017-05-04    2
15.jpg                         Walmart                    None  2016-07-22    0
16.jpg                         Walmart                   23.19  2017-11-13    3
17.jpg                         Walmart                   38.68  2017-01-16    1
18.jpg                         Wal

## Stage 3 — `POST /receipt/parse` (Python FastAPI)

Exercises the endpoint added in `python-api/main.py`. The service must be running at `PYTHON_API_URL`.

In [15]:
# ── Health check ──────────────────────────────────────────────────────────────
try:
    health = requests.get(f"{PYTHON_API_URL}/", timeout=5)
    print(f"Python API reachable — status {health.status_code}")
except requests.exceptions.ConnectionError:
    raise RuntimeError(
        f"Cannot reach Python API at {PYTHON_API_URL}.\n"
        "Start it with: cd python-api && uvicorn main:app --reload"
    )

# ── Call /receipt/parse for each image's tokens ───────────────────────────────
api_results: dict[str, dict] = {}

for name, tokens in ocr_results.items():
    resp = requests.post(
        f"{PYTHON_API_URL}/receipt/parse",
        json={"tokens": tokens},
        timeout=30,
    )
    print(f"{'─'*60}")
    print(f"[{name}]  HTTP {resp.status_code}")

    if resp.status_code == 503:
        print("  ⚠  503: receipt_parser not loaded by the API (check startup logs)")
        continue

    resp.raise_for_status()
    body = resp.json()
    api_results[name] = body

    if not body.get("success"):
        print(f"  ✗  error: {body.get('error')}")
        continue

    r = body["result"]
    print(f"  store   : {r['store']}")
    print(f"  date    : {r['date']}")
    print(f"  total   : {r['total']}")
    print(f"  subtotal: {r['subtotal']}")
    print(f"  taxes   : {r['taxes']}")
    print(f"  items ({len(r['items'])})")
    for item in r["items"]:
        print(f"    qty={item['quantity']}  price={item['price']:<7}  {item['name']}")

# ── Consistency check: API vs local parser ────────────────────────────────────
print()
print("Consistency (API total == local parser total):")
for name in ocr_results:
    if name not in api_results:
        print(f"  {name}: SKIPPED (API call failed)")
        continue
    api_total   = api_results[name]["result"]["total"]
    local_total = parsed_results[name]["total"]
    match = (api_total == local_total) or (
        api_total is not None and local_total is not None
        and abs(api_total - local_total) < 0.01
    )
    status = "✓" if match else "✗"
    print(f"  {status}  {name}  api={api_total}  local={local_total}")

Python API reachable — status 200
────────────────────────────────────────────────────────────
[0.jpg]  HTTP 200
  store   : Unknown
  date    : None
  total   : None
  subtotal: None
  taxes   : []
  items (3)
    qty=1  price=0.41     1783 ST# 5748 OP# 00000158 TEIL14 TR# 03178 BANANAS OOOOOOOO4011KF
    qty=1  price=0.2      Ib /0.49
    qty=1  price=13.8     CASH TEND 88 CHANGE' DUE
────────────────────────────────────────────────────────────
[1.jpg]  HTTP 200
  store   : Trader Joe's
  date    : 2014-06-28
  total   : 840.0
  subtotal: 838.68
  taxes   : [{'rate': 0.0, 'amount': 0.49}, {'rate': 0.0, 'amount': 838.68}]
  items (4)
    qty=1  price=0.49     98
    qty=1  price=2.49     Eo29EENut Butter
    qty=1  price=1.69     CREAMY SALTED
    qty=1  price=838.68   @ 0.69
────────────────────────────────────────────────────────────
[10.jpg]  HTTP 200
  store   : SPAR
  date    : 2021-02-23
  total   : 338.16
  subtotal: None
  taxes   : []
  items (11)
    qty=1  price=16.99    Mi

## Stage 4 — `POST /api/receipt/process` (Next.js → Supabase)

Sends the structured receipt to the Next.js route added in `app/api/receipt/process/route.ts`.
This writes rows to `product_mappings`, `ingredient_match_queue`, and `pantry_items`.

> **Auth**: set `CLERK_SESSION_TOKEN` in the Config cell above.

In [ ]:
if not CLERK_SESSION_TOKEN:
    print(
        "⚠  CLERK_SESSION_TOKEN is empty — /api/receipt/process will return 401.\n"
        "   Set it in the Config cell (copy __session cookie from your browser).\n"
        "   Continuing anyway so you can see the raw response."
    )

# ── Health check ──────────────────────────────────────────────────────────────
try:
    requests.get(f"{NEXTJS_API_URL}/", timeout=5)
    print(f"Next.js dev server reachable at {NEXTJS_API_URL}")
except requests.exceptions.ConnectionError:
    raise RuntimeError(
        f"Cannot reach Next.js at {NEXTJS_API_URL}.\n"
        "Start it with: npm run dev"
    )

process_results: dict[str, dict] = {}

for name, parsed in parsed_results.items():
    print(f"\n{'─'*60}")
    print(f"[{name}]")

    if not parsed["items"]:
        print("  No items extracted — skipping")
        continue

    payload = {
        "parsedReceipt": {
            "store"   : parsed["store"],
            "date"    : parsed["date"],
            "items"   : parsed["items"],
            "subtotal": parsed["subtotal"],
            "total"   : parsed["total"],
        }
    }

    resp = requests.post(
        f"{NEXTJS_API_URL}/api/receipt/process",
        json=payload,
        headers={
            "Content-Type": "application/json",
            "Cookie": f"__session={CLERK_SESSION_TOKEN}",
        },
        timeout=60,
    )

    print(f"  HTTP {resp.status_code}")

    if resp.status_code == 401:
        print("  ✗  Unauthorized — set CLERK_SESSION_TOKEN in the Config cell")
        continue

    body = resp.json()
    process_results[name] = body

    if not body.get("success"):
        print(f"  ✗  error: {body.get('error')}")
        continue

    print(f"  store_brand  : {body['storeBrand']}")
    print(f"  receipt_date : {body['receiptDate']}")
    print(f"  pantry_added : {body['pantryAdded']}")
    print(f"  queued       : {body['queued']}")
    print(f"  skipped      : {body['skipped']}")
    print(f"  items ({len(body['items'])})")
    for item in body["items"]:
        sid  = item.get("standardized_ingredient_id", "")[:8] + "…" if item.get("standardized_ingredient_id") else ""
        mid  = item.get("mapping_id", "")[:8] + "…" if item.get("mapping_id") else ""
        pid  = item.get("pantry_item_id", "")[:8] + "…" if item.get("pantry_item_id") else ""
        print(f"    [{item['status']:<7}]  {item['name'][:40]:<40}  pantry={pid}  mapping={mid}  ingredient={sid}")

## Stage 4b — Dry-run with synthetic payload

Uses a hand-crafted receipt so you can test Stage 4 without any sample images.

In [ ]:
SYNTHETIC_RECEIPT = {
    "parsedReceipt": {
        "store": "Walmart",
        "date": "2026-03-29",
        "items": [
            {"name": "MILK WHOLE 1GAL",       "quantity": 1, "price": 3.49},
            {"name": "EGGS LG 12CT",           "quantity": 1, "price": 2.78},
            {"name": "BREAD WHITE 20OZ",       "quantity": 2, "price": 1.98},
            {"name": "COMPLETELY MADE UP ITEM", "quantity": 1, "price": 9.99},
            {"name": "",                        "quantity": 1, "price": 0.00},  # should be skipped
        ],
        "subtotal": 20.22,
        "total": 20.22,
    }
}

print("Sending synthetic Walmart receipt to /api/receipt/process ...\n")

resp = requests.post(
    f"{NEXTJS_API_URL}/api/receipt/process",
    json=SYNTHETIC_RECEIPT,
    headers={
        "Content-Type": "application/json",
        "Cookie": f"__session={CLERK_SESSION_TOKEN}",
    },
    timeout=60,
)

print(f"HTTP {resp.status_code}")
body = resp.json()
pprint.pprint(body)

if body.get("success"):
    # Basic assertions
    total_processed = body["pantryAdded"] + body["queued"] + body["skipped"]
    input_items = len(SYNTHETIC_RECEIPT["parsedReceipt"]["items"])

    assert total_processed == input_items, (
        f"Expected {input_items} total items processed, got {total_processed}"
    )

    empty_items = [i for i in body["items"] if i["name"] == ""]
    assert all(i["status"] == "skipped" for i in empty_items), \
        "Empty-name items should always be skipped"

    non_empty = [i for i in body["items"] if i["name"]]
    assert all(i["status"] in {"added", "queued"} for i in non_empty), \
        "Non-empty items must be 'added' or 'queued', not 'skipped'"

    queued_items = [i for i in body["items"] if i["status"] == "queued"]
    assert all(i.get("pantry_item_id") for i in queued_items), \
        "Queued items must still get a pantry_item_id"

    print("\n✓ All assertions passed")
else:
    print(f"\n✗ Request failed: {body.get('error')}")

## Summary

In [ ]:
print("Pipeline Test Summary")
print("=" * 65)

print(f"\nStage 1 — easyOCR")
for name, tokens in ocr_results.items():
    print(f"  ✓  {name}: {len(tokens)} tokens")

print(f"\nStage 2 — receipt_parser (local)")
for name, result in parsed_results.items():
    has_items = len(result["items"]) > 0
    print(f"  {'✓' if has_items else '⚠'}  {name}: store={result['store']}, {len(result['items'])} items, total={result['total']}")

print(f"\nStage 3 — POST /receipt/parse (FastAPI)")
for name in ocr_results:
    if name in api_results and api_results[name].get("success"):
        n = len(api_results[name]["result"]["items"])
        print(f"  ✓  {name}: {n} items")
    else:
        print(f"  ✗  {name}: call failed or skipped")

print(f"\nStage 4 — POST /api/receipt/process (Next.js → Supabase)")
for name in parsed_results:
    if name in process_results and process_results[name].get("success"):
        r = process_results[name]
        print(f"  ✓  {name}: +{r['pantryAdded']} pantry, {r['queued']} queued, {r['skipped']} skipped")
    else:
        print(f"  ✗  {name}: call failed or skipped (check auth token)")

print("\nDone.")